# Munti — does the no-positions model actually use word order?

Removing the positional embeddings cost only 0.039 nats (val 1.5051 -> 1.5444)
and the text stayed coherent. Two explanations fit that:

1. the model ignores word order and TinyStories is predictable enough from a
   bag of recent words, or
2. the model still knows the order, having recovered it some other way.

A causal decoder can do (2): token 5 attends over a 5-token prefix and token 50
over a 50-token prefix, so prefix length is itself a positional signal. That's a
documented result (Haviv et al. 2022), but it's a claim about *this* model, so
measure it rather than cite it.

**The probe:** score each model on ordinary validation text, then on the same
text with the tokens within each window shuffled. A model that ignores order is
barely affected by shuffling. A model that uses order gets much worse.

Identical batches and a fixed seed for both models, so the only difference is
the weights.

In [ ]:
import os, shutil, sys, glob, json
from pathlib import Path

WORK = Path("/kaggle/working/munti-repo")
if not WORK.exists():
    roots = [Path(p).parent for p in glob.glob("/kaggle/input/*/**/pyproject.toml", recursive=True)]
    assert roots, "repo not found"
    shutil.copytree(roots[0], WORK)
os.chdir(WORK); sys.path.insert(0, str(WORK))

(WORK / "data").mkdir(exist_ok=True)
for name in ("val.bin", "tokenizer.json"):
    src = next((p for p in glob.glob(f"/kaggle/input/*/**/data/{name}", recursive=True)), None)
    assert src, f"{name} not found"
    shutil.copy(src, WORK / "data" / name)

# Both checkpoints: the baseline from munti-train, the ablation from munti-ablation.
CKPTS = {}
for p in glob.glob("/kaggle/input/*/**/ckpt.pt", recursive=True):
    CKPTS["nopos" if "nopos" in p else "baseline"] = p
print(json.dumps(CKPTS, indent=2))
assert len(CKPTS) == 2, "need both kernels attached as sources"

In [ ]:
import torch
from munti import data as D
from munti.model import Munti

device = "cuda"
val = D.load_split("val")
BATCH, BLOCK, ITERS = 64, 256, 100

# Build the evaluation batches once, so every model sees identical data.
g = torch.Generator().manual_seed(1234)
batches = [D.get_batch(val, BATCH, BLOCK, "cpu", generator=g) for _ in range(ITERS)]

def shuffle_within(x, y, seed):
    """Permute tokens within each sequence, keeping (input, target) aligned.

    The targets must be permuted the same way: the task stays 'predict the next
    token of this shuffled sequence'. Shuffling only the inputs would measure
    something else entirely (predicting original text from scrambled context).
    """
    gg = torch.Generator().manual_seed(seed)
    full = torch.cat([x, y[:, -1:]], dim=1)          # the original T+1 tokens
    out = torch.stack([f[torch.randperm(f.numel(), generator=gg)] for f in full])
    return out[:, :-1], out[:, 1:]

@torch.no_grad()
def score(model, shuffled=False):
    model.eval()
    tot = 0.0
    for i, (x, y) in enumerate(batches):
        if shuffled:
            x, y = shuffle_within(x, y, seed=i)
        _, loss = model(x.to(device), y.to(device))
        tot += loss.item()
    return tot / len(batches)

rows = {}
for name, path in sorted(CKPTS.items()):
    m = Munti.from_checkpoint(torch.load(path, map_location=device, weights_only=False), device=device)
    normal, shuf = score(m), score(m, shuffled=True)
    rows[name] = {"params": m.num_params(), "learned_pos": m.cfg.learned_pos,
                  "val": round(normal, 4), "val_shuffled": round(shuf, 4),
                  "delta": round(shuf - normal, 4)}
    print(name, rows[name])

Path("out-eval").mkdir(exist_ok=True)
Path("out-eval/order_sensitivity.json").write_text(json.dumps(rows, indent=2))
print(json.dumps(rows, indent=2))

In [ ]:
# A second, blunter check: greedy continuations from both models on the same
# prompt, plus what each does with a prompt whose words are scrambled.
from munti.sample import generate_text
from munti import tokenizer as tk

tok = tk.load()
lines = []
for name, path in sorted(CKPTS.items()):
    m = Munti.from_checkpoint(torch.load(path, map_location=device, weights_only=False), device=device)
    for prompt in ["Once upon a time, there was a little girl named Lily.",
                   "girl a Lily. little named was there time, a upon Once"]:
        txt = generate_text(m, tok, prompt, device=device, max_new_tokens=120, temperature=0.8, top_k=200)
        lines.append(f"### {name} | prompt: {prompt!r}\n\n> {txt}\n")
        print(lines[-1][:600])
Path("out-eval/order_samples.md").write_text("\n".join(lines), encoding="utf-8")
shutil.make_archive("/kaggle/working/munti-eval", "zip", "out-eval")